In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge, ElasticNet
from category_encoders import TargetEncoder  # pip install category_encoders
import warnings
warnings.filterwarnings('ignore')

In [2]:
# -----------------------------
# 1. Загрузка данных
# -----------------------------
train = pd.read_csv('/kaggle/input/home-data-for-ml-course/train.csv')
test = pd.read_csv('/kaggle/input/home-data-for-ml-course/test.csv')

In [3]:
test_ids = test['Id']
y = train['SalePrice']
X = train.drop(['SalePrice', 'Id'], axis=1)
X_test = test.drop('Id', axis=1)

In [4]:
# -----------------------------
# 2. Очистка данных: удаление шумных столбцов
# -----------------------------
# Удаляем столбцы с >80% пропусков
high_na_cols = X.columns[X.isna().mean() > 0.8]
X.drop(high_na_cols, axis=1, inplace=True)
X_test.drop(high_na_cols, axis=1, inplace=True)

In [5]:
# -----------------------------
# 3. Feature Engineering: создание новых признаков
# -----------------------------
def create_features(df):
    # Общая площадь дома
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    # Количество ванных комнат (с весом для half-bath)
    df['Bathrooms'] = df['FullBath'] + 0.5 * df['HalfBath']
    # Возраст ремонта (0, если не ремонтировался)
    df['YearRemodAdd'] = np.where(df['YearRemodAdd'] == df['YearBuilt'], 0, df['YearRemodAdd'] - df['YearBuilt'])
    # Бинарные флаги
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['NewHouse'] = (df['YearBuilt'] >= 2000).astype(int)
    return df

X = create_features(X)
X_test = create_features(X_test)

In [6]:
# -----------------------------
# 4. Определение типов признаков
# -----------------------------
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Заполнение пропусков
X[categorical_features] = X[categorical_features].fillna('Missing')
X_test[categorical_features] = X_test[categorical_features].fillna('Missing')


X[numeric_features] = X[numeric_features].fillna(X[numeric_features].median())
X_test[numeric_features] = X_test[numeric_features].fillna(X[numeric_features].median())

In [7]:
# -----------------------------
# 5. Пайплайн предобработки: Target Encoding для категориальных
# -----------------------------
categorical_transformer = TargetEncoder(smoothing=1.0)  # Лучше, чем OrdinalEncoder


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Логарифмирование целевой переменной
y_log = np.log1p(y)

# Разделение на train/val
X_train, X_val, y_train_log, y_val_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

In [8]:
# -----------------------------
# 6. Модели: RandomForest, XGBoost, LightGBM, CatBoost, ElasticNet
# -----------------------------
models = {}

print("Обучаем RandomForest...")
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])
pipeline_rf.fit(X_train, y_train_log)
models['RandomForest'] = pipeline_rf


print("Обучаем XGBoost...")
pipeline_xg = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='rmse'
    ))
])
pipeline_xg.fit(X_train, y_train_log)
models['XGBoost'] = pipeline_xg

print("Обучаем LightGBM...")
pipeline_lg = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=10,
        random_state=42,
        n_jobs=-1
    ))
])
pipeline_lg.fit(X_train, y_train_log)
models['LightGBM'] = pipeline_lg

print("Обучаем CatBoost...")
pipeline_cb = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', CatBoostRegressor(
        iterations=3000,
        learning_rate=0.01,
        depth=6,
        verbose=0,
        random_state=42
    ))
])
pipeline_cb.fit(X_train, y_train_log)
models['CatBoost'] = pipeline_cb

print("Обучаем ElasticNet...")
pipeline_en = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42))
])
pipeline_en.fit(X_train, y_train_log)
models['ElasticNet'] = pipeline_en

Обучаем RandomForest...
Обучаем XGBoost...
Обучаем LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002772 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3451
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 75
[LightGBM] [Info] Start training from score 12.030658
Обучаем CatBoost...
Обучаем ElasticNet...


In [9]:
# 7. Оценка моделей на валидации
# -----------------------------
print("\nОценка моделей на валидационной выборке...")
val_predictions = {}
for name, model in models.items():
    pred_log = model.predict(X_val)
    rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_val_log), np.expm1(pred_log)))
    val_predictions[name] = pred_log
    print(f"RMSLE {name}: {rmsle:.4f}")


Оценка моделей на валидационной выборке...
RMSLE RandomForest: 0.1454
RMSLE XGBoost: 0.1330
RMSLE LightGBM: 0.1366
RMSLE CatBoost: 0.1269
RMSLE ElasticNet: 0.1951


In [10]:
# -----------------------------
# 8. Предсказание на тестовой выборке и формирование сабмишенов
# -----------------------------
print("\nФормирование финальных сабмишенов...")


# Сохраняем сабмишены для всех моделей
submission_files = {}
for name in models.keys():
    preds = np.expm1(models[name].predict(X_test))
    submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
    filename = f'submission_{name.lower().replace(" ", "_")}.csv'
    submission.to_csv(filename, index=False)
    submission_files[name] = filename
    print(f"Сабмишен {name} сохранён как '{filename}'")


Формирование финальных сабмишенов...
Сабмишен RandomForest сохранён как 'submission_randomforest.csv'
Сабмишен XGBoost сохранён как 'submission_xgboost.csv'
Сабмишен LightGBM сохранён как 'submission_lightgbm.csv'
Сабмишен CatBoost сохранён как 'submission_catboost.csv'
Сабмишен ElasticNet сохранён как 'submission_elasticnet.csv'


In [11]:
# -----------------------------
# 11. Итоговые метрики
# -----------------------------
print("\nИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ")
print("=" * 40)
for name in models.keys():
    pred_log = models[name].predict(X_val)
    rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_val_log), np.expm1(pred_log)))
    print(f"{name}: {rmsle:.4f}")
print("=" * 40)

# -----------------------------
# 12. Описание подхода (для README или отчёта)
# -----------------------------
print("\nОПИСАНИЕ ПОДХОДА:")
print("- Очистка данных: удалены столбцы с >80% пропусков и нулевой дисперсией.")
print("- Feature Engineering: созданы признаки TotalSF, Bathrooms, YearRemodAdd, HasPool, HasGarage, NewHouse.")
print("- Кодирование: TargetEncoder для категориальных признаков (лучше учитывает связь с целевой переменной).")
print("- Модели: RandomForest, XGBoost, LightGBM, CatBoost, ElasticNet.")


ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ
RandomForest: 0.1454
XGBoost: 0.1330
LightGBM: 0.1366
CatBoost: 0.1269
ElasticNet: 0.1951

ОПИСАНИЕ ПОДХОДА:
- Очистка данных: удалены столбцы с >80% пропусков и нулевой дисперсией.
- Feature Engineering: созданы признаки TotalSF, Bathrooms, YearRemodAdd, HasPool, HasGarage, NewHouse.
- Кодирование: TargetEncoder для категориальных признаков (лучше учитывает связь с целевой переменной).
- Модели: RandomForest, XGBoost, LightGBM, CatBoost, ElasticNet.
